# RAG: Baseline vs Re-Ranking

**Document:** ICC Men's T20 World Cup 2026 Playing Conditions (106 pages, Feb 2026)

- **Pipeline 1:** bi-encoder → top 10 by cosine → top 3 → LLM → Answer 1
- **Pipeline 2:** same top 10 → cross-encoder re-rank → top 3 → LLM → Answer 2

Chunks come in two versions with the same boundaries: **A** = plain text, **B** = section title + text. Both pipelines are run on both.

## Setup

In [1]:
import os, re, random, hashlib, urllib.request
from pathlib import Path

ROOT = Path.cwd()
DATA, OUT = ROOT / "data", ROOT / "outputs"
DATA.mkdir(exist_ok=True); OUT.mkdir(exist_ok=True)
os.environ.setdefault("HF_HOME", str(ROOT / "models" / "hf"))  # keep model downloads in the project
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")

import pymupdf
import pandas as pd
from transformers import AutoTokenizer
from transformers.utils import logging as hf_logging
hf_logging.disable_progress_bar()

pd.set_option("display.max_colwidth", 140)
random.seed(0)

## Phase 1: Chunking

### 1.1 Load PDF
Not committed (ICC copyright), so it's downloaded from the ICC site if missing.

In [2]:
PDF_URL = "https://images.icc-cricket.com/image/upload/prd/yyt7m8uh9c1uehrbvuwb.pdf"
PDF_PATH = DATA / "icc_t20wc_2026_playing_conditions.pdf"
PDF_SHA256 = "0a5af6c6bd1677c835a4b7a0c81ebb68a1b0894c87ef0f158348dc84e58f323b"

if not PDF_PATH.exists():
    urllib.request.urlretrieve(PDF_URL, PDF_PATH)
sha = hashlib.sha256(PDF_PATH.read_bytes()).hexdigest()
assert sha == PDF_SHA256, f"PDF differs from the version used in this study: {sha}"

doc = pymupdf.open(PDF_PATH)
print(f"{PDF_PATH.name}: {doc.page_count} pages, {PDF_PATH.stat().st_size / 1e6:.1f} MB, sha256 OK")

icc_t20wc_2026_playing_conditions.pdf: 106 pages, 0.8 MB, sha256 OK


### 1.2 Raw text
Each page starts with a page number and a running header. Clause numbers are followed by a tab (`9.1\t`), and section titles are in caps.

In [3]:
print(repr(doc[15].get_text()[:700]))

"16\nPlaying Conditions  |  ICC Men's T20 World Cup 2026\n9\t\nPREPARATION AND MAINTENANCE OF \nTHE PLAYING AREA\n9.1\t\nRolling\nThe pitch shall not be rolled during the match except as permitted in clauses 9.1.1 and\xa09.1.2.\n9.1.1\t\nFrequency and duration of rolling During the match the pitch may be rolled at the \nrequest of the captain of the side batting second, for a period of not more than \n7\xa0minutes, before the start of the second innings.\n9.1.2\t\nRolling after a delayed start In addition to the rolling permitted above, if, after the toss \nand before the first innings of the match, the start is delayed, the captain of the batting \nside may request that the pitch be rolled for not more than 7 minute"


### 1.3 Clean
- Skip the cover (p.1), the contents pages (p.3, p.72) and Appendix E's blank calculation forms (p.98–99).
- Drop page numbers, headers and diagram labels. Labels are set in 7.9–9.2 pt and body text in 10 pt, so anything under 9.5 pt goes.

In [4]:
SKIP_PAGES = {1, 3, 72, 98, 99}   # cover, contents, appendix contents, Appendix E forms
HEADER_RE = re.compile(r"^\s*(\d{1,3}\s+)?(ICC Men's T20 World Cup 2026\s*\|\s*Playing Conditions"
                       r"|Playing Conditions\s*\|\s*ICC Men's T20 World Cup 2026)\s*$")

def small_text(page, max_size=9.5):
    # lines set in a small font: diagram labels, header, page number
    return {"".join(s["text"] for s in l["spans"]).strip()
            for b in page.get_text("dict")["blocks"] for l in b.get("lines", [])
            if l["spans"] and max(s["size"] for s in l["spans"]) < max_size}

def extract_lines(doc):
    lines = []                                   # (page_number, line)
    for pno, page in enumerate(doc, start=1):
        if pno in SKIP_PAGES:
            continue
        raw = page.get_text().replace("\xa0", " ").replace("\x07", "")
        small = small_text(page)
        for i, line in enumerate(raw.splitlines()):
            if i == 0 and line.strip() == str(pno):          # page number
                continue
            if HEADER_RE.match(line) or not line.strip() or re.search(r"_{3,}", line) or line.strip() in small:
                continue
            lines.append((pno, line.rstrip(" ")))    # keep TABs: they mark clause numbers
    return lines

lines = extract_lines(doc)
print(f"{len(lines)} lines kept from {doc.page_count - len(SKIP_PAGES)} pages")

5040 lines kept from 101 pages


### 1.4 Parse clauses
Walk through the lines, keeping track of the current appendix and section. Every clause number starts a new unit.

A clause title on its own line gets a colon, so it doesn't run into the rule: `9.1 Rolling: The pitch…`.

In [5]:
CLAUSE_RE = re.compile(r"^(\d{1,2}(?:\.\d{1,2})+|\d{1,2}\.)(?:\t\s*(.*)|\s*)$")
SECTION_RE = re.compile(r"^(\d{1,2})(?:\t\s*(.*)|\s*)$")
APPENDIX_RE = re.compile(r"^APPENDIX ([A-J]):\s*(.*)$")

def is_caps(line):
    letters = [c for c in line if c.isalpha()]
    return len(letters) >= 3 and all(c.isupper() for c in letters)

def read_caps_title(lines, j, first, stop_re):
    title = [first]
    while (j < len(lines) and is_caps(lines[j][1]) and not stop_re.match(lines[j][1])
           and not APPENDIX_RE.match(lines[j][1].strip())):
        title.append(lines[j][1].strip()); j += 1
    return " ".join(t for t in title if t), j

def parse_units(lines):
    units, appendix, section, cur = [], None, ("0", "PREAMBLE"), None
    def new(clause, pno, first=""):
        return {"appendix": appendix, "section": section, "clause": clause, "page": pno,
                "lines": [first] if first.strip() else [], "title_line": not first.strip()}
    i = 0
    while i < len(lines):
        pno, line = lines[i]
        nxt = lines[i + 1][1] if i + 1 < len(lines) else ""
        m_app, m_sec, m_cl = APPENDIX_RE.match(line.strip()), SECTION_RE.match(line), CLAUSE_RE.match(line)
        if m_app:
            units.append(cur)
            title, i = read_caps_title(lines, i + 1, m_app.group(2).strip(), SECTION_RE)
            appendix = (m_app.group(1), title)
            section = appendix
            cur = new(None, pno)
        elif m_sec and not m_cl and (is_caps(m_sec.group(2) or "") or (not (m_sec.group(2) or "").strip() and is_caps(nxt))):
            units.append(cur)
            title, i = read_caps_title(lines, i + 1, (m_sec.group(2) or "").strip(), CLAUSE_RE)
            section = (m_sec.group(1), title)
            cur = new(None, pno)
        elif m_cl and section[0].isdigit() and section[0] != "0" and "." in m_cl.group(1).rstrip(".") \
                and not m_cl.group(1).startswith(section[0] + "."):
            cur["lines"].append(line)                 # a number inside a table (e.g. "4.5" in 28.7), not a clause
            i += 1
        elif m_cl:
            units.append(cur)
            cur = new(m_cl.group(1).rstrip("."), pno, m_cl.group(2) or "")
            i += 1
        else:
            cur = cur or new(None, pno)               # preamble before the first section
            cur["lines"].append(line)
            i += 1
    units.append(cur)
    return [u for u in units if u and u["lines"]]

def join_lines(ls):
    t = "\n".join(l.strip() for l in ls if l.strip())
    t = re.sub(r"-\n(?=[a-z])", "-", t)              # "wicket-\nkeeper" -> "wicket-keeper"
    return re.sub(r"\s+", " ", t.replace("\t", " ").replace("\n", " ")).strip()

def clause_text(u):
    ls = [l for l in u["lines"] if l.strip()]
    if not u["clause"]:
        return join_lines(ls)
    first = ls[0].strip()
    # title on its own line: short, no end punctuation, next line starts a sentence
    if (u["title_line"] and len(ls) > 1 and len(first.split()) <= 8 and not re.search(r"[.;:,]$", first)
            and re.match(r"[A-Z0-9(–‘“\"]", ls[1].strip())):
        return f"{u['clause']} {first}: {join_lines(ls[1:])}"
    return f"{u['clause']} {join_lines(ls)}"

units = parse_units(lines)
for u in units:
    u["text"] = clause_text(u)
print(f"{len(units)} clause units")
pd.DataFrame([{"clause": u["clause"], "section": " ".join(u["section"]), "page": u["page"], "text": u["text"][:100]}
              for u in units[60:66]])

1200 clause units


,clause,section,page,text
0,2.7.1,2 THE UMPIRES,7,"2.7.1 It is solely for the umpires together to decide whether either conditions of ground, weather o"
1,2.7.2,2 THE UMPIRES,7,2.7.2 Conditions shall be regarded as dangerous if there is actual and foreseeable risk to the safet
2,2.7.3,2 THE UMPIRES,7,"2.7.3 Conditions shall be regarded as unreasonable if, although posing no risk to safety, it would n"
3,2.7.4,2 THE UMPIRES,7,2.7.4 If the umpires consider the ground is so wet or slippery as to deprive the bowler of a reasona
4,2.8,2 THE UMPIRES,7,2.8 Suspension of play in dangerous or unreasonable circumstances
5,2.8.1,2 THE UMPIRES,7,2.8.1 All references to ground include the pitch. See clause 6.1 (Area of pitch).


### 1.5 Chunk
- Merge consecutive clauses from the same section, up to ~180 tokens (MiniLM's limit is 256).
- Split clauses over 220 tokens by sentence, with 1 sentence of overlap. Later pieces start with `<clause> (cont.)`.
- Never leave a heading alone at the end of a chunk.

Chunks are kept small on purpose: with big chunks the answer is almost always in the top 3 already, and re-ranking has nothing to fix.

In [6]:
TARGET, MAX = 180, 220
tok = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
ntok = lambda s: len(tok.tokenize(s))
for u in units:
    u["ntok"] = ntok(u["text"])

def split_sentences(text):
    sents = []
    for s in re.split(r"(?<=[.;:])\s+(?=[A-Z0-9(])", text):
        words = s.split()
        while ntok(" ".join(words)) > TARGET:    # very long sentence
            k = len(words) // 2
            while ntok(" ".join(words[:k])) > TARGET: k //= 2
            while k < len(words) and ntok(" ".join(words[:k + 10])) <= TARGET: k += 10
            sents.append(" ".join(words[:k])); words = words[k:]
        sents.append(" ".join(words))
    return sents

def split_long(u):
    out, win = [], []
    for s in split_sentences(u["text"]):
        if win and ntok(" ".join(win + [s])) > TARGET:
            out.append(" ".join(win))
            win = [win[-1]] if ntok(win[-1]) < TARGET // 3 else []
        win.append(s)
    out.append(" ".join(win))
    if u["clause"]:
        out = out[:1] + [f"{u['clause']} (cont.) {t}" for t in out[1:]]
    return [dict(u, text=t, ntok=ntok(t)) for t in out]

def pack(units):
    pieces = [p for u in units for p in (split_long(u) if u["ntok"] > MAX else [u])]
    chunks, buf = [], []
    def emit():
        clauses = [b["clause"] for b in buf if b["clause"]]
        chunks.append({"appendix": buf[0]["appendix"], "section": buf[0]["section"], "page": buf[0]["page"],
                       "clauses": (f"{clauses[0]}–{clauses[-1]}" if len(set(clauses)) > 1 else (clauses[0] if clauses else "")),
                       "text": " ".join(b["text"] for b in buf)})
    for p in pieces:
        same = buf and (p["appendix"], p["section"]) == (buf[0]["appendix"], buf[0]["section"])
        size = sum(b["ntok"] for b in buf)
        if buf and not (same and size < 15) and (not same or size + p["ntok"] > TARGET):
            carry = [buf.pop()] if same and len(buf) > 1 and buf[-1]["ntok"] < 15 else []   # bare heading
            emit(); buf = carry
        buf.append(p)
    emit()
    return chunks

chunks = pack(units)
print(f"{len(units)} clause units -> {len(chunks)} chunks")

1200 clause units -> 440 chunks


### 1.6 Two versions
`text` (A) and `text_titled` (B = `[section] ` + text). Same IDs in both.

In [7]:
SMALL = {"and", "of", "the", "for", "to", "on", "in", "a", "an", "or", "by"}
def nice(s):
    return " ".join(w if w.startswith("(") and w.strip("()").isupper() and len(w.strip("()")) <= 4
                    else w.lower() if k and w.lower() in SMALL else w.capitalize()
                    for k, w in enumerate(s.split()))

for k, c in enumerate(chunks):
    c["id"] = f"c{k:03d}"
    sec_num, sec_title = c["section"]
    path = f"Appendix {c['appendix'][0]}: {nice(c['appendix'][1])}" if c["appendix"] else ""
    if not c["appendix"] or sec_title != c["appendix"][1]:
        path = (path + " > " if path else "") + (nice(sec_title) if sec_num == "0" else f"{sec_num} {nice(sec_title)}")
    c["section_path"] = path
    c["text_titled"] = f"[{path}] {c['text']}"
    c["ntok"], c["ntok_titled"] = ntok(c["text"]), ntok(c["text_titled"])

df = pd.DataFrame(chunks)[["id", "section_path", "clauses", "page", "ntok", "ntok_titled", "text", "text_titled"]]
df.head(3)

,id,section_path,clauses,page,ntok,ntok_titled,text,text_titled
0,c000,Preamble,,2,165,170,PREAMBLE – THE SPIRIT OF CRICKET Cricket owes much of its appeal and enjoyment to the fact that it should be played not only according t...,[Preamble] PREAMBLE – THE SPIRIT OF CRICKET Cricket owes much of its appeal and enjoyment to the fact that it should be played not only ...
1,c001,Preamble,,2,81,86,"Congratulate the opposition on their successes, and enjoy those of your own team. Thank the officials and your opposition at the end of ...","[Preamble] Congratulate the opposition on their successes, and enjoy those of your own team. Thank the officials and your opposition at ..."
2,c002,1 The Players,1.1–1.2.1,4,137,142,"1.1 Number of players: A match is played between two sides, each of eleven players, one of whom shall be captain. 1.2 Nomination and rep...","[1 The Players] 1.1 Number of players: A match is played between two sides, each of eleven players, one of whom shall be captain. 1.2 No..."


### 1.7 Save

In [8]:
CHUNKS_PATH = DATA / "chunks.jsonl"
df.to_json(CHUNKS_PATH, orient="records", lines=True, force_ascii=False)
print(f"saved {len(df)} chunks -> {CHUNKS_PATH.relative_to(ROOT)}")

saved 440 chunks -> data/chunks.jsonl


### 1.8 Sanity checks

In [9]:
print(df[["ntok", "ntok_titled"]].describe().round(1).T[["count", "min", "25%", "50%", "75%", "max"]])
print(f"\nchunks over 256 tokens: plain {(df.ntok > 256).sum()}, titled {(df.ntok_titled > 256).sum()}")
print(f"main-body chunks: {(~df.section_path.str.startswith('Appendix')).sum()}, appendix chunks: {df.section_path.str.startswith('Appendix').sum()}")

             count   min    25%    50%    75%    max
ntok         440.0   4.0  115.0  148.0  168.0  215.0
ntok_titled  440.0  11.0  122.8  155.0  175.0  223.0

chunks over 256 tokens: plain 0, titled 0
main-body chunks: 324, appendix chunks: 116


In [10]:
# chunks per part
part = df.section_path.where(df.section_path.str.startswith("Appendix"), "Main body (sections 1-42)").str.split(" > ").str[0]
part.value_counts().sort_index().to_frame("chunks")

,chunks
section_path,
Appendix A: Definitions,28
Appendix B: Equipment,12
Appendix C: The Venue,3
Appendix D: Decision Review System (DRS),55
Appendix F: Super Over Procedure,9
Appendix G: Application of the Reserve Day for the Semi-finals and Final,4
Appendix H: Use of Electronic Communications Equipment,1
Appendix I: On-field Injury Assessments and Treatment,1
Appendix J: Summary of Changes to Playing Conditions for Warm-up Matches,3


In [11]:
# a few random chunks
for _, r in df.sample(3, random_state=1).iterrows():
    print(f"{r.id} | {r.section_path} | clauses {r.clauses} | p.{r.page} | {r.ntok} tokens\n{r.text}\n")

c283 | 41 Unfair Play | clauses 41.5.10–41.6.1 | p.62 | 141 tokens
41.5.10 The umpires may then report the matter to the ICC Match Referee who shall take such action as is considered appropriate against the fielder concerned. 41.6 Bowling of dangerous and unfair short pitched deliveries 41.6.1 Notwithstanding clause 41.6.2, the bowling of short pitched deliveries is dangerous if the bowler’s end umpire considers that, taking into consideration the skill of the striker, by their speed, length, height and direction they are likely to inflict physical injury on him. The fact that the striker is wearing protective equipment shall be disregarded. In the first instance the umpire decides that the bowling of short pitched deliveries has become dangerous under clause 41.6.1

c369 | Appendix D: Decision Review System (DRS) > 1 General | clauses 1.1.5 | p.88 | 158 tokens
1.1.5 The table below summarises the minimum requirements for DRS to be used, and the regulations around the appointment of th

Some chunks to keep in mind for later:
- 9.1: the rolling rule stays together with its exceptions
- Follow-on: the whole chunk is "Shall not apply.", so it can only be found in version B
- Main rules say 11 players, warm-ups (App. J) say max 15 → keyword trap

In [12]:
show = lambda mask: df[mask][["id", "section_path", "clauses", "page", "text"]]
pd.concat([
    show(df.text.str.contains("shall not be rolled")),
    show(df.text.str.fullmatch("Shall not apply.")),
    show(df.text.str.startswith("1.1 Number of players") | df.text.str.contains("Each side shall consist of a maximum 15 players")),
])

,id,section_path,clauses,page,text
54,c054,9 Preparation and Maintenance of the Playing Area,9.1–9.1.2,16,9.1 Rolling: The pitch shall not be rolled during the match except as permitted in clauses 9.1.1 and 9.1.2. 9.1.1 Frequency and duration...
92,c092,14 The Follow-on,,24,Shall not apply.
93,c093,15 Declaration and Forfeiture,,24,Shall not apply.
2,c002,1 The Players,1.1–1.2.1,4,"1.1 Number of players: A match is played between two sides, each of eleven players, one of whom shall be captain. 1.2 Nomination and rep..."
437,c437,Appendix J: Summary of Changes to Playing Conditions for Warm-up Matches,1.1,105,NOTE: ICC Men’s T20 World Cup 2026 warm-up matches will not be official Twenty 20 Internationals. ICC Match Referees will not be on duty...


In [13]:
r = df[df.section_path.str.contains("Follow-on")].iloc[0]
print("A:", r.text)
print("B:", r.text_titled)

A: Shall not apply.
B: [14 The Follow-on] Shall not apply.


## Phase 2: Pipeline 1 (baseline retrieval)

### 2.1 Load chunks

In [14]:
import time
import numpy as np
import torch, faiss
from sentence_transformers import SentenceTransformer

chunks = pd.read_json(DATA / "chunks.jsonl", lines=True)
print(len(chunks), "chunks")

440 chunks


### 2.2 Bi-encoders
Two bi-encoders, both reported:
- `all-MiniLM-L6-v2`: the standard small model
- `bge-small-en-v1.5`: stronger, same size (384 dims). Queries get its recommended instruction prefix.

In [15]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BI_ENCODERS = {
    "minilm": ("sentence-transformers/all-MiniLM-L6-v2", ""),
    "bge": ("BAAI/bge-small-en-v1.5", "Represent this sentence for searching relevant passages: "),
}
VERSIONS = {"A": "text", "B": "text_titled"}

encoders = {name: SentenceTransformer(model_id, device=DEVICE) for name, (model_id, _) in BI_ENCODERS.items()}
print("device:", DEVICE)

device: cpu


### 2.3 Embed chunks and build indexes
Embeddings are normalised, so inner product = cosine similarity. `IndexFlatIP` does an exact search over all 445 chunks. That gives 4 indexes: 2 models × 2 versions.

In [16]:
indexes = {}
for name, enc in encoders.items():
    for v, col in VERSIONS.items():
        t0 = time.perf_counter()
        emb = enc.encode(chunks[col].tolist(), batch_size=64, normalize_embeddings=True, convert_to_numpy=True)
        indexes[(name, v)] = faiss.IndexFlatIP(emb.shape[1])
        indexes[(name, v)].add(emb)
        print(f"{name:6} {v}: {emb.shape} in {time.perf_counter() - t0:.2f}s")

minilm A: (440, 384) in 8.95s


minilm B: (440, 384) in 8.67s


bge    A: (440, 384) in 16.70s


bge    B: (440, 384) in 16.62s


### 2.4 Retrieve top 10

In [17]:
def retrieve(query, model="minilm", version="A", k=10):
    prefix = BI_ENCODERS[model][1]
    q = encoders[model].encode([prefix + query], normalize_embeddings=True, convert_to_numpy=True)
    scores, idx = indexes[(model, version)].search(q, k)
    hits = chunks.iloc[idx[0]][["id", "section_path", "clauses", "text"]].reset_index(drop=True)
    hits.insert(0, "rank", range(1, k + 1))
    hits.insert(2, "cosine", scores[0].round(3))
    return hits

### 2.5 Question set
40 questions in `eval/questions.csv`, 4 tiers of 10, each tier harder than the last:
1. **direct**: uses the rulebook's own wording
2. **paraphrase**: everyday fan wording
3. **exception / negation**: "can a batter…", "is it allowed…"
4. **keyword trap / cross-ref / multi-chunk**: warm-up vs main rules, Super Over appendix, answers split over chunks

Gold chunks are stored as anchor phrases (a unique bit of the rule text, or `§section`), so the labels still work if the chunking changes.

In [18]:
qs = pd.read_csv(ROOT / "eval" / "questions.csv")

def resolve(anchors):
    ids = []
    for a in anchors.split(" || "):
        m = chunks.section_path == a[1:] if a.startswith("§") else chunks.text.str.contains(a, regex=False)
        assert m.any(), f"anchor not found: {a}"
        ids += chunks[m].id.tolist()
    return ids

qs["gold"] = qs.gold_anchors.map(resolve)
qs.groupby("tier").type.value_counts().unstack(fill_value=0)

type,cross-ref,direct,exception,keyword-trap,multi-chunk,negation,paraphrase,table
tier,,,,,,,,
1,0,10,0,0,0,0,0,0
2,0,0,0,0,0,0,10,0
3,0,0,5,0,0,5,0,0
4,3,0,0,5,1,0,0,1


### 2.6 Baseline ranks
Rank of the best gold chunk among all 446, for every question and setup:
- **≤ 3**: the LLM gets it
- **4–10**: missed, but the re-ranker can still rescue it
- **> 10**: lost, because the re-ranker only sees the top 10

In [19]:
def gold_ranks(model, version):
    prefix = BI_ENCODERS[model][1]
    q = encoders[model].encode([prefix + x for x in qs.question], normalize_embeddings=True, convert_to_numpy=True)
    _, idx = indexes[(model, version)].search(q, len(chunks))
    ids = chunks.id.to_numpy()[idx]
    return [min(list(row).index(g) + 1 for g in gold) for row, gold in zip(ids, qs.gold)]

SETUPS = [(m, v) for m in BI_ENCODERS for v in VERSIONS]
ranks = qs[["qid", "tier", "type", "question"]].copy()
for m, v in SETUPS:
    ranks[f"{m}-{v}"] = gold_ranks(m, v)
ranks

,qid,tier,type,question,minilm-A,minilm-B,bge-A,bge-B
0,q01,1,direct,How many players are there on each side in a World Cup match?,1,1,3,2
1,q02,1,direct,How many substitute fielders can a captain nominate for a World Cup match?,1,2,1,2
2,q03,1,direct,What is the minimum over rate to be achieved in a match?,1,1,1,1
3,q04,1,direct,How many overs can one bowler bowl in a full 20-over innings?,1,1,1,1
4,q05,1,direct,How long can the pitch be rolled before the start of the second innings?,1,1,1,1
5,q06,1,direct,How many unsuccessful player review requests is each team allowed per innings?,1,1,1,1
6,q07,1,direct,For how many overs do the Powerplay fielding restrictions apply?,2,2,1,1
7,q08,1,direct,How many fast short-pitched deliveries can a bowler bowl in an over?,1,1,2,2
8,q09,1,direct,What is the minimum distance from the boundary rope to the first solid object?,1,1,1,1
9,q10,1,direct,How long is the interval between innings?,2,2,1,1


In [20]:
def zone(r):
    return "top 3" if r <= 3 else ("4-10" if r <= 10 else ">10")

summary = pd.concat({s: ranks.groupby("tier")[s].apply(lambda x: x.map(zone).value_counts()).unstack(fill_value=0)
                     for s in [f"{m}-{v}" for m, v in SETUPS]}, axis=1)
summary.reindex(columns=["top 3", "4-10", ">10"], level=1)   # questions per zone, per tier

minilm-A          minilm-B          bge-A          bge-B         
        top 3 4-10 >10    top 3 4-10 >10 top 3 4-10 >10 top 3 4-10 >10
tier                                                                  
1          10    0   0       10    0   0    10    0   0    10    0   0
2           8    0   2        7    1   2     8    0   2     8    0   2
3           4    4   2        3    5   2     6    2   2     7    1   2
4           4    2   4        5    1   4     4    5   1     5    4   1

In [21]:
# example: a tier 4 question
q = qs[qs.qid == "q34"].iloc[0]
print(q.question, "| gold:", q.gold)
retrieve(q.question, "minilm", "A").assign(text=lambda d: d.text.str[:90])

In the one-over eliminator, if a team already used up both its reviews in the match, can it still review? | gold: ['c425']


,rank,id,cosine,section_path,clauses,text
0,1,c411,0.488,Appendix D: Decision Review System (DRS) > 3 Player Review,3.6–3.6.3,"3.6 Number of Player Review requests permitted 3.6.1 In each innings, each team shall be a"
1,2,c392,0.486,Appendix D: Decision Review System (DRS) > 3 Player Review,3.2.4–3.2.6,"3.2.4 No replays, either at normal speed or slow motion, shall be shown on a big screen to"
2,3,c420,0.473,Appendix D: Decision Review System (DRS) > 3 Player Review,3.9–3.9.1,3.9 Combining Umpire Review with Player Review 3.9.1 If an Umpire Review (under paragraph
3,4,c388,0.417,Appendix D: Decision Review System (DRS) > 3 Player Review,3.1–3.1.2,The following paragraphs shall operate in addition to and in conjunction with paragraph 2
4,5,c414,0.414,Appendix D: Decision Review System (DRS) > 3 Player Review,3.6.8,3.6.8 A Player Review categorised as ‘Unsuccessful’ may be reinstated by the ICC Match Ref
5,6,c415,0.413,Appendix D: Decision Review System (DRS) > 3 Player Review,3.6.9–3.6.10,3.6.9 The third umpire shall be responsible for counting the number Player Reviews categor
6,7,c390,0.408,Appendix D: Decision Review System (DRS) > 3 Player Review,3.2.2,3.2.2 The total time elapsed between the ball becoming dead and the review request being m
7,8,c412,0.406,Appendix D: Decision Review System (DRS) > 3 Player Review,3.6.4–3.6.5,3.6.4 Where a request for a Player Review of an LBW decision results in the on-field decis
8,9,c389,0.390,Appendix D: Decision Review System (DRS) > 3 Player Review,3.1.3–3.2.1,3.1.3 Only the batter involved in a dismissal may request a Player Review of an Out decisi
9,10,c391,0.379,Appendix D: Decision Review System (DRS) > 3 Player Review,3.2.3,"3.2.3 The captain may consult with the bowler and other fielders, and the two batters may"


## Phase 3: Pipeline 2 (re-ranking) + LLM answers

### 3.1 Cross-encoder
`cross-encoder/ms-marco-MiniLM-L6-v2` reads each (question, chunk) pair from the bi-encoder's top 10 together and gives it a relevance score. The 10 are then re-sorted by that score.

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2", device=DEVICE)
by_id = chunks.set_index("id")

def rerank(query, hits, version="A"):
    texts = by_id.loc[hits.id, VERSIONS[version]].tolist()
    scores = reranker.predict([(query, t) for t in texts], show_progress_bar=False)
    out = hits.assign(base_rank=hits["rank"], ce_score=np.round(scores, 3))
    out = out.sort_values("ce_score", ascending=False).reset_index(drop=True)
    out["rank"] = range(1, len(out) + 1)
    return out

### 3.2 Both pipelines on all questions
For each question and setup: baseline top 10, re-ranked top 10, and the gold chunk's rank in each (NaN = not in the top 10).

In [ ]:
rows = []
for m, v in SETUPS:
    for q in qs.itertuples():
        base = retrieve(q.question, m, v)
        rr = rerank(q.question, base, v)
        best = lambda h: h.loc[h.id.isin(q.gold), "rank"].min()
        rows.append({"qid": q.qid, "tier": q.tier, "setup": f"{m}-{v}",
                     "base_top10": list(base.id), "rr_top10": list(rr.id), "ce_scores": list(rr.ce_score),
                     "gold_base": best(base), "gold_rr": best(rr)})
runs = pd.DataFrame(rows)
runs.to_csv(OUT / "retrieval_runs.csv", index=False)

def summarise(df):
    return pd.Series({
        "hit@3 baseline": (df.gold_base <= 3).mean(),
        "hit@3 reranked": (df.gold_rr <= 3).mean(),
        "MRR@10 baseline": (1 / df.gold_base).fillna(0).mean(),
        "MRR@10 reranked": (1 / df.gold_rr).fillna(0).mean(),
        "fixed (4-10 -> top 3)": ((df.gold_base > 3) & (df.gold_rr <= 3)).sum(),
        "broken (top 3 -> 4-10)": ((df.gold_base <= 3) & (df.gold_rr > 3)).sum(),
    })

runs.groupby("setup").apply(summarise).round(3)

In [ ]:
runs.groupby(["setup", "tier"]).apply(summarise)[["hit@3 baseline", "hit@3 reranked", "fixed (4-10 -> top 3)", "broken (top 3 -> 4-10)"]].unstack("setup").round(2)

### 3.3 LLM
`Qwen3-4B-Instruct-2507` on its own GPU, fp16, greedy decoding (same input → same answer).
- The context shows each chunk's section path in both versions, so A vs B only differs in *which* chunks were retrieved.
- A quick sum checks that fp16 works on the 2080 Ti. If it fails, the model is reloaded in fp32 across two GPUs.

In [ ]:
from transformers import AutoModelForCausalLM

LLM_ID = "Qwen/Qwen3-4B-Instruct-2507"
LLM_DEVICE = "cuda:1" if torch.cuda.device_count() > 1 else DEVICE
llm_tok = AutoTokenizer.from_pretrained(LLM_ID, padding_side="left")
llm = AutoModelForCausalLM.from_pretrained(LLM_ID, dtype=torch.float16, device_map=LLM_DEVICE).eval()

def generate(conversations, max_new_tokens=160, batch_size=8):
    outs = []
    for i in range(0, len(conversations), batch_size):
        prompts = [llm_tok.apply_chat_template(c, tokenize=False, add_generation_prompt=True) for c in conversations[i:i + batch_size]]
        enc = llm_tok(prompts, return_tensors="pt", padding=True).to(llm.device)
        with torch.no_grad():
            out = llm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                               temperature=None, top_p=None, top_k=None, pad_token_id=llm_tok.pad_token_id)
        outs += llm_tok.batch_decode(out[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
    return [o.strip() for o in outs]

check = generate([[{"role": "user", "content": "What is 17 + 25? Reply with just the number."}]])[0]
if "42" not in check:   # fp16 overflow -> fall back to fp32 split over two GPUs
    del llm; torch.cuda.empty_cache()
    llm = AutoModelForCausalLM.from_pretrained(LLM_ID, dtype=torch.float32, device_map="auto",
                                               max_memory={1: "10GiB", 2: "10GiB"}).eval()
    check = generate([[{"role": "user", "content": "What is 17 + 25? Reply with just the number."}]])[0]
print("check:", check, "| dtype:", llm.dtype)

### 3.4 Prompts
- **No RAG:** the question alone, to see what the model already knows (the PDF is newer than its training data).
- **RAG:** the same prompt for Answer 1 and Answer 2. Answer only from the context, or say "Not in the context."

In [ ]:
NO_RAG_SYSTEM = ("You answer questions about the ICC Men's T20 World Cup 2026 Playing Conditions from your own knowledge. "
                 "If you don't know, say so. Answer in 1-3 sentences.")
RAG_SYSTEM = ("You answer questions about the ICC Men's T20 World Cup 2026 Playing Conditions. "
              "Use only the context. If the context doesn't contain the answer, say \"Not in the context.\" "
              "Answer in 1-3 sentences.")

def no_rag_messages(question):
    return [{"role": "system", "content": NO_RAG_SYSTEM}, {"role": "user", "content": question}]

def rag_messages(question, ids):
    ctx = "\n\n".join(f"[{k}] {by_id.at[i, 'section_path']}\n{by_id.at[i, 'text']}" for k, i in enumerate(ids, 1))
    return [{"role": "system", "content": RAG_SYSTEM}, {"role": "user", "content": f"Context:\n{ctx}\n\nQuestion: {question}"}]

print(rag_messages(qs.question[0], runs.base_top10[0][:3])[1]["content"][:600])

### 3.5 Generate answers
Three answers per question and setup: no RAG, Answer 1 (baseline top 3), Answer 2 (re-ranked top 3). Identical prompts are generated once. Answers are cached in `outputs/llm_cache.json`, so re-running the notebook doesn't regenerate them.

In [ ]:
import json
CACHE = OUT / "llm_cache.json"
cache = json.loads(CACHE.read_text()) if CACHE.exists() else {}
key = lambda msgs: hashlib.sha256(json.dumps(msgs).encode()).hexdigest()[:16]

def cached_generate(conversations, **kw):
    todo = sorted({key(c): c for c in conversations if key(c) not in cache}.items(), key=lambda kv: len(json.dumps(kv[1])))
    if todo:
        t0 = time.perf_counter()
        for (k, _), ans in zip(todo, generate([c for _, c in todo], **kw)):
            cache[k] = ans
        CACHE.write_text(json.dumps(cache, indent=0))
        print(f"generated {len(todo)} in {time.perf_counter() - t0:.0f}s")
    return [cache[key(c)] for c in conversations]

question = qs.set_index("qid").question
plan = [(q.qid, "none", "no_rag", no_rag_messages(q.question), "") for q in qs.itertuples()]
for r in runs.itertuples():
    for mode, top in [("baseline", r.base_top10[:3]), ("reranked", r.rr_top10[:3])]:
        plan.append((r.qid, r.setup, mode, rag_messages(question[r.qid], top), ",".join(top)))

texts = cached_generate([p[3] for p in plan])
answers = pd.DataFrame([{"qid": qid, "setup": s, "mode": mode, "context": ctx, "answer": a}
                        for (qid, s, mode, _, ctx), a in zip(plan, texts)])
print(len(answers), "answers")

### 3.6 Grading
Each answer is compared with the expected answer from `eval/questions.csv`:
- the same LLM grades it: correct = 1, partial = 0.5, wrong = 0
- "Not in the context" / "I don't know" counts as **abstain** (0, but not a hallucination)
- **hallucination** = a wrong answer given confidently

In [ ]:
JUDGE_SYSTEM = ("You grade answers about cricket rules. Compare the candidate answer with the reference answer. "
                "Reply with exactly one word: correct (same meaning as the reference), "
                "partial (partly right or missing a key condition) or wrong.")
ABSTAIN = re.compile(r"not in the context|don't know|do not know|not sure|cannot determine|no information", re.I)

def judge_messages(qid, ans):
    q = qs.set_index("qid").loc[qid]
    return [{"role": "system", "content": JUDGE_SYSTEM},
            {"role": "user", "content": f"Question: {q.question}\nReference answer: {q.expected_answer}\nCandidate answer: {ans}"}]

answers["abstain"] = answers.answer.str.contains(ABSTAIN)
verdicts = cached_generate([judge_messages(r.qid, r.answer) for r in answers.itertuples()], max_new_tokens=4)
def parse_verdict(v):
    v = v.lower()
    if "partial" in v: return "partial"
    if "incorrect" in v or "wrong" in v: return "wrong"
    return "correct" if "correct" in v else "wrong"

answers["verdict"] = [parse_verdict(v) for v in verdicts]
answers.loc[answers.abstain, "verdict"] = "abstain"
answers["score"] = answers.verdict.map({"correct": 1, "partial": 0.5}).fillna(0)
answers["hallucination"] = answers.verdict.eq("wrong")
answers = answers.merge(qs[["qid", "tier"]], on="qid")
answers.to_csv(OUT / "answers.csv", index=False)
answers.verdict.value_counts()

In [ ]:
# score by mode (no RAG has one row per question, RAG modes have one per setup)
answers.groupby(["mode", "setup"]).agg(score=("score", "mean"), hallucination=("hallucination", "mean"),
                                       abstain=("abstain", "mean")).round(2)

In [ ]:
# score by tier: no RAG vs Answer 1 vs Answer 2 (MiniLM-A)
sel = answers[(answers.setup == "minilm-A") | (answers["mode"] == "no_rag")]
sel.pivot_table(index="tier", columns="mode", values="score", aggfunc="mean").round(2)[["no_rag", "baseline", "reranked"]]

## Phase 5: Latency
Average over the 40 questions after 5 warm-up runs, for MiniLM-A on GPU and on CPU. `torch.cuda.synchronize()` makes the GPU timings include all queued work. The last rows answer the scaling question: running the cross-encoder on all 446 chunks vs the bi-encoder search.

In [ ]:
def sync():
    for d in range(torch.cuda.device_count()):
        torch.cuda.synchronize(d)

def bench(fn, inputs, warmup=5, gpu=False):
    for x in inputs[:warmup]:
        fn(x)
    times = []
    for x in inputs:
        if gpu: sync()
        t0 = time.perf_counter(); fn(x)
        if gpu: sync()
        times.append((time.perf_counter() - t0) * 1000)
    return round(float(np.mean(times)), 2), round(float(np.std(times)), 2)

questions = qs.question.tolist()
index = indexes[("minilm", "A")]
top10 = {q: retrieve(q, "minilm", "A").id.tolist() for q in questions}
lat = []

def pipeline2(q, embed, ce):
    _, idx = index.search(embed(q), 10)
    ids = chunks.id.to_numpy()[idx[0]]
    return ce.predict([(q, by_id.at[i, "text"]) for i in ids], show_progress_bar=False)

for dev in (["cuda"] if torch.cuda.is_available() else []) + ["cpu"]:
    gpu = dev == "cuda"
    enc = encoders["minilm"] if gpu else SentenceTransformer(BI_ENCODERS["minilm"][0], device="cpu")
    ce = reranker if gpu else CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2", device="cpu")
    embed = lambda q: enc.encode([q], normalize_embeddings=True, convert_to_numpy=True)
    qemb = {q: embed(q) for q in questions}
    stages = {
        "query embedding": lambda q: embed(q),
        "vector search (top 10)": lambda q: index.search(qemb[q], 10),
        "re-rank 10 pairs": lambda q: ce.predict([(q, by_id.at[i, "text"]) for i in top10[q]], show_progress_bar=False),
        "Pipeline 1 retrieval": lambda q: index.search(embed(q), 10),
        "Pipeline 2 retrieval": lambda q: pipeline2(q, embed, ce),
        "cross-encoder on all 446 chunks": lambda q: ce.predict([(q, t) for t in chunks.text], batch_size=64, show_progress_bar=False),
    }
    for name, fn in stages.items():
        inputs = questions[:10] if name.startswith("cross-encoder on all") else questions
        mean, std = bench(fn, inputs, warmup=2 if name.startswith("cross-encoder on all") else 5, gpu=gpu)
        lat.append({"device": dev, "stage": name, "mean ms": mean, "std ms": std})

In [ ]:
# LLM answer time (GPU, Answer 1 prompt, 10 questions, not cached)
llm_inputs = [rag_messages(q, top10[q][:3]) for q in questions[:10]]
mean, std = bench(lambda c: generate([c]), llm_inputs, warmup=2, gpu=True)
lat.append({"device": "cuda", "stage": "LLM answer", "mean ms": mean, "std ms": std})
latency = pd.DataFrame(lat)
latency.to_csv(OUT / "latency.csv", index=False)
latency.pivot(index="stage", columns="device", values="mean ms").reindex(
    ["query embedding", "vector search (top 10)", "re-rank 10 pairs", "Pipeline 1 retrieval", "Pipeline 2 retrieval",
     "cross-encoder on all 446 chunks", "LLM answer"])